## Token Intervention with Generation Afterwards

In [1]:
%load_ext autoreload
%autoreload 2

### Overview

Injects a counterfactual number into a truncated reasoning chain, then lets the model freely generate the rest of its reasoning to see whether it reacts to the injected value.

### Set-up

In [ ]:
import sys
sys.path.append("/nas/ucb/daniel_d_kang/arithmetic-reasoning-causality/src")

import torch
import gc
from tqdm import tqdm

import _config

/nas/ucb/daniel_d_kang/arithmetic-reasoning-causality/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
print(torch.cuda.is_available())

True


In [4]:
prompt_config = _config.PromptConfig(
    model_type="GPT-OSS_stepwise", # GPT-OSS or R1
    prompt_type="h", # empty or pre_result or pre_sum
    data_root="/nas/ucb/daniel_d_kang/arithmetic-reasoning-causality/data",
)
run_config = _config.RunConfig(
    experiment_root="/nas/ucb/daniel_d_kang/arithmetic-reasoning-causality/experiments/token_intervention",
    output_filename=f"generation{prompt_config.suffix}.csv",
)
batch_size = 24
max_new_tokens = 512
resume_from_base_1_num = 188
resume_from_base_2_num = 321

model, tokenizer = _config.load_model(prompt_config.model_type)

Loading OSS model on device: auto


Loading weights: 100%|██████████| 411/411 [00:02<00:00, 180.01it/s]


In [5]:
divided_prompts = _config.load_divided_prompts(prompt_config)
print(f"loaded {len(divided_prompts)} divided prompts")

loaded 3072 divided prompts


In [6]:
def get_generation_intervention_prompt(row):
    return row['base_before'] + str(row['source_number'])

# Batch code below uses _config.build_number_prompts for the same construction.

### Run intervention + free generation

For each divided prompt, swap in the counterfactual number and let the model freely generate the rest of its reasoning, recording the full continuation.

In [7]:
# Get header of divided prompts dataset
header = list(divided_prompts.columns) + ['intervention_prompt', 'generated_text']
filepath = _config.build_run_output_filepath(prompt_config, run_config, header)

skip = True
for i in tqdm(range(0, len(divided_prompts), batch_size)):
    torch.cuda.empty_cache()
    gc.collect()
    batch_rows = divided_prompts.iloc[i:i+batch_size]
    if batch_rows['base_1_num'].iloc[0] == resume_from_base_1_num and batch_rows['base_2_num'].iloc[0] == resume_from_base_2_num:
        skip = False
    if skip:
        continue
    
    # Prepare batch of intervention prompts
    intervention_prompts = _config.build_number_prompts(batch_rows, "source_number")
    
    # Tokenize all prompts in the batch
    tokenized_inputs = tokenizer(intervention_prompts, add_special_tokens=False, return_tensors="pt", padding=True, padding_side="left").to(model.device)
    
    # Generate for the entire batch
    generations = model.generate(
        tokenized_inputs.input_ids,
        attention_mask=tokenized_inputs.attention_mask,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    
    # Process each generated text in the batch
    for j, (_, row) in enumerate(batch_rows.iterrows()):
        intervention_prompt = intervention_prompts[j]
        generated_text = tokenizer.decode(generations[j]).replace(intervention_prompt, "").replace(tokenizer.pad_token[-1], "")
        _config.write_to_csv(filepath, row.to_list() + [intervention_prompt, generated_text])


100%|██████████| 128/128 [1:36:20<00:00, 45.16s/it] 


### Diagnostics

In [7]:
from collections import Counter

print("model.device:", model.device)
print("hf_device_map:", getattr(model, "hf_device_map", None))

param_devices = Counter(str(p.device) for p in model.parameters())
print(param_devices)

for name, p in model.named_parameters():
    if p.device.type == "cpu":
        print(name, p.shape, p.dtype, p.device)

model.device: cuda:0
hf_device_map: {'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 0, 'model.layers.10': 0, 'model.layers.11': 1, 'model.layers.12': 1, 'model.layers.13': 1, 'model.layers.14': 1, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.norm': 1, 'model.rotary_emb': 1, 'lm_head': 'disk'}
Counter({'cuda:1': 222, 'cuda:0': 188, 'meta': 1})
